In [33]:
import Gmsh: gmsh
import Pkg; Pkg.add("Distributions")
using Gridap, GridapGmsh
using Random
using Distributions

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [54]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square");
    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);
    gmsh.model.geo.addLine(1, 2, 1); 
    gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3, 4, 3);
    gmsh.model.geo.addLine(4, 1, 4);
    
    gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

    gmsh.model.geo.addPlaneSurface([1], 1);

    gmsh.model.geo.synchronize();

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")

    gmsh.model.mesh.generate(2); # dimension is 2
    
    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [55]:
h = 0.03; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00247404s, CPU 0.003134s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.27419s, CPU 0.274168s)
Info    : 1441 nodes 2884 elements


TrialFESpace()

In [56]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [68]:
n=6
Δt, T = 2.0^(-n), 2*π/2 # time step, final time
num_steps = Int(round(T / Δt))

201

In [69]:
f(x) = x[1]*(1-x[1]) * x[2]*(1-x[2])
g = x->f(x) 

#104 (generic function with 1 method)

In [82]:
u₀ = x -> 0.0
w₀ = x -> 2.0

#135 (generic function with 1 method)

In [83]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

# LHS
A = M + (Δt/4)*K

1305×1305 SparseArrays.SparseMatrixCSC{Float64, Int64} with 8861 stored entries:
⎡⡑⣬⡁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣺⡾⠬⢿⣷⣶⠆⠈⣩⣛⣆⢢⠨⠄⡘⢗⣿⣷⢞⎤
⎢⠁⠈⠻⣦⣀⠀⡀⠀⠈⠎⠐⡆⠀⡈⠐⠄⠀⠤⠀⠄⠀⠀⠛⠐⠈⠢⢀⢀⢈⡼⣾⡋⠁⠆⠀⠃⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠘⠿⣧⡅⠀⢑⡥⡀⠁⠀⠂⠈⠀⠀⠦⠀⠀⠀⠀⠀⠀⠭⠀⠀⣮⡊⢷⠪⢅⣦⣽⣣⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠁⠉⣿⣿⣴⠌⣀⠁⠠⠁⠀⠀⠀⠀⠀⠰⠀⠀⠈⠂⠁⢀⢨⠹⣟⣳⠤⢘⣡⣍⣷⠄⠅⠀⠀⠀⎥
⎢⠀⠀⡢⠄⠕⡴⡐⠟⠻⣦⣟⣝⢍⡚⠠⠂⠰⠣⠂⠔⠀⠀⠀⠄⠤⡉⢉⡘⢾⠾⣼⣵⣩⠏⠪⠒⠀⠀⠀⠀⎥
⎢⠀⠀⠰⠤⠄⠈⠄⠘⣟⢽⣿⢟⡷⠸⣯⣮⡴⡿⠀⠀⠀⠀⢄⡇⠀⠈⠝⢾⢋⡱⡼⠼⢿⣹⡘⡩⢐⡀⠀⠄⎥
⎢⠀⠀⡀⠠⠠⠀⠄⠂⣣⠱⣙⡋⣻⣾⡷⢋⢻⡹⣂⠰⣄⠀⠂⢈⠧⡚⠺⠦⡴⠮⠞⣾⣪⡵⣍⣽⡔⠀⢁⡁⎥
⎢⠀⠀⠐⠄⠂⠀⠀⠀⠠⠂⡫⣿⡽⢋⡻⣮⣿⣊⣚⠷⣬⠀⠷⠂⠧⡠⣘⠄⠯⠗⠬⢎⣵⣶⡺⢽⠀⠁⠁⠁⎥
⎢⠀⠀⠀⡄⠠⡄⠀⠀⠴⡂⣴⡯⣟⡲⡻⢻⢵⣷⢯⡷⡚⢉⣛⡷⡻⣞⣰⣾⢼⣽⢞⣟⣵⣷⣡⣜⠩⣉⠈⠉⎥
⎢⠀⠀⠀⠄⠀⠀⢀⡀⢈⠄⠀⠀⢈⡘⢾⡜⢯⡷⠱⢆⣩⡶⢧⡦⣌⣥⢺⡦⣽⣼⡍⣙⣧⣿⣻⡍⡹⠲⠖⠀⎥
⎢⣠⣠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠂⠛⡞⢈⢣⡾⠻⣦⣔⡐⡉⡁⣼⠁⠐⣠⠴⢺⠋⢿⡿⠛⢲⣼⣧⣱⎥
⎢⡚⡏⢛⠀⠀⠀⠢⠀⠀⠄⠤⠵⡈⢀⠹⠃⢿⡼⠩⡷⢐⠹⢿⢗⣾⡻⢜⠻⠸⠼⢋⠛⢣⠤⠂⡏⠳⢟⣽⡥⎥
⎢⢿⣷⠢⡀⠃⠃⠁⢀⡄⠣⡀⠀⣩⠣⠉⡣⣻⢮⠆⣽⠇⠨⣾⡻⢛⢔⡰⠁⡔⠑⣉⡧⢽⡽⠛⣺⣦⡪⢍⢵⎥
⎢⠸⠟⠀⢐⡠⣤⣆⡒⣃⠰⣳⣅⠺⡆⠒⠜⣰⣾⠺⡶⠖⠛⣶⡑⠔⠊⢱⣶⢔⢀⢴⢔⣄⣶⢤⢢⢊⢛⡾⠏⎥
⎢⡆⣠⣂⡴⢮⣌⢿⣹⣺⡗⢏⡰⡰⡏⢯⠇⣖⣷⣓⣿⠐⣠⣒⡆⢔⠉⠐⢑⣵⣿⣷⢾⢧⢟⣵⠌⠠⣒⣄⢁⎥
⎢⠻⢼⡾⠻⠎⢆⣀⢃⢖⣿⣒⡏⣺⣥⡢⢇⣾⢵⣇⢩⣰⣃⣯⠐⠧⡼⢐⢗⣹⣟⢿⢗⡷⣽⡛⢮⠀⡸⠪⡂⎥
⎢⡈⡒⠡⠄⣌⣿⡅⢾⡧⠞⣟⣳⢎⡾⢱⣿⢵⣿⣭⣿⣯⣄⠉⡖⣗⡷⢠⣽⣭⢗⣝⣯⣿⣿⣿⠻⠠⣫⠁⠈⎥
⎢⣀⠡⠤⠀⠉⠚⠙⠟⢪⠂⡖⡨⣇⣽⣞⣎⣁⢾⡟⠾⣿⠋⡬⠤⣻⣠⠠⣓⡑⠟⡻⣌⣿⡛⣿⣿⣁⠈⠛⠀⎥
⎢⣽⣵⠀⠀⠀⠀⠁⠁⠀⠀⠐⠰⠐⠉⠄⠀⡇⢢⢳⡊⣘⣶⣽⢆⡨⡻⣮⢐⢠⢢⣀⡠⡤⣢⡁⠘⢿⣷⠲⣶⎥
⎣⣹⢟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠄⠅⠰⠅⠀⡆⠀⠘⠁⢍⣻⠗⡿⢇⣕⡾⠏⠄⢙⠪⠢⡁⠀⠛⠀⢸⣦⣵⢟⎦

In [84]:
uh0 = interpolate_everywhere(u₀, U)   # U^0
uh1 = interpolate_everywhere(x -> u₀(x) + Δt * w₀(x), U)   # U^1

SingleFieldFEFunction():
 num_cells: 2744
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 429604364652079094

In [85]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    pvd[0.0] = createvtk(Ω, "tmp/stochasticwave2_0.vtu",
                         cellfields=["uh" => uh0])
    pvd[Δt]  = createvtk(Ω, "tmp/stochasticwave2_1.vtu",
                         cellfields=["uh" => uh1])

    # dof-vectors
    uvec_prev = get_free_dof_values(uh0)  # U^{n-1}
    uvec_curr = get_free_dof_values(uh1)  # U^{n}

    for n in 1:(num_steps-1)
        t_n = n*Δt
        Δβ = rand(Normal(0,sqrt(Δt)))
        b(v) = ∫(g*v)dΩ
        Fn = assemble_vector(b, V)
        noise = Δt * Δβ * Fn
        
        # RHS
        rhs = (2*M - ((Δt)^2/2)*K) * uvec_curr -
              (M + ((Δt)^2/4)*K) * uvec_prev + noise

        # Solve for next step
        uvec_next = A \ rhs
        uh_next = FEFunction(U, uvec_next) # solve for next value

        t_next = (n+1)*Δt
        vtufilename = "tmp/stochasticwave2_$(n+1).vtu"
        pvd[t_next] = createvtk(Ω, vtufilename, cellfields=["uh" => uh_next])

        #update
        uvec_prev = uvec_curr
        uvec_curr = uvec_next
    end
end

203-element Vector{String}:
 "wave.pvd"
 "tmp/stochasticwave2_0.vtu"
 "tmp/stochasticwave2_1.vtu"
 "tmp/stochasticwave2_2.vtu"
 "tmp/stochasticwave2_3.vtu"
 "tmp/stochasticwave2_4.vtu"
 "tmp/stochasticwave2_5.vtu"
 "tmp/stochasticwave2_6.vtu"
 "tmp/stochasticwave2_7.vtu"
 "tmp/stochasticwave2_8.vtu"
 "tmp/stochasticwave2_9.vtu"
 "tmp/stochasticwave2_10.vtu"
 "tmp/stochasticwave2_11.vtu"
 ⋮
 "tmp/stochasticwave2_190.vtu"
 "tmp/stochasticwave2_191.vtu"
 "tmp/stochasticwave2_192.vtu"
 "tmp/stochasticwave2_193.vtu"
 "tmp/stochasticwave2_194.vtu"
 "tmp/stochasticwave2_195.vtu"
 "tmp/stochasticwave2_196.vtu"
 "tmp/stochasticwave2_197.vtu"
 "tmp/stochasticwave2_198.vtu"
 "tmp/stochasticwave2_199.vtu"
 "tmp/stochasticwave2_200.vtu"
 "tmp/stochasticwave2_201.vtu"